# test-C — C3: Docling chunking + VLM figure descriptions

answering: **gpt-4o-mini**, figure descriptions: **gpt-4o-mini**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-C"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,e0a036e5-97db-4fc5-9431-8bcffb4dd204,478,gpt-4o-mini,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,c9ed513f-cbb6-4a76-a53d-f1f1a73bbc6d,285,gpt-4o-mini,text-embedding-3-small
2,Kohler Generator Manual.pdf,bffdf9b4-503c-4249-8927-08dc1e88f919,294,gpt-4o-mini,text-embedding-3-small
3,Nidec Motor Manual.pdf,de5f8bbf-be39-40e5-9532-71bf2b22f2b4,116,gpt-4o-mini,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,befc5148-1919-447f-813f-56975d0048e0,119,gpt-4o-mini,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-C tables          30  0.756  0.944  1.000            0.861          1.000          0.956           1.000         0.683             0.879           0.944            0.831      NaN
       cross_document  30  0.856  0.933  0.989            0.908          1.000          0.900           1.000         0.727             0.904           0.972            0.889      NaN
       topical         30  0.700  0.878  0.922            0.798          0.967          0.967           0.967         0.838             0.762           0.833            0.819      NaN
       paraphrase      35  0.562  0.810  0.905            0.695          0.990          0.905           0.971         0.736             0.800           0.881            0.800      NaN
       figures         25  0.787  0.907  0.947            0.852          0.987          0.973           0.947         0.804             0.684           0.606            0.567      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN      1.0

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-C,170,0.724,0.891,0.951,0.818,0.989,0.938,0.978,0.756,0.81,0.856,0.789,1.0


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.960,0.859,0.808
2,0.940,0.850,0.760
3,0.953,0.859,0.798


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    1.0
2    1.0
3    1.0
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,The context does not specify a frequency or sp...
1,"In Fig. 27 of the Kohler KD27V12 manual, appro...",figures,Kohler Generator Man,0.0,The context provided does not specify the exac...
2,"In the ABB induction machine manual, what is t...",tables,ABB Manual for Induc,0.0,The Pt-100 resistance at exactly 100°C is 164....
3,How does ABB calculate equivalent operating ho...,topical,ABB Manual for Induc,0.0,The provided context does not specify how ABB ...
4,"From Figure 42 of the ABB manual, at 18 brushe...",figures,ABB Manual for Induc,0.0,The approximate maximum cleaning interval for ...
5,How does ABB calculate equivalent operating ho...,topical,ABB Manual for Induc,0.0,The provided context does not include informat...
6,"From Figure 38 of the ABB manual, approximatel...",figures,ABB Manual for Induc,0.0,The coefficient Kt at a winding temperature of...
7,"From Figure 38 of the ABB manual, approximatel...",figures,ABB Manual for Induc,0.0,To find the coefficient Kt at a winding temper...
8,What is the correct charge generator belt tens...,figures,EMSA Diesel Generato,0.0,The correct charge generator belt tension on t...
9,What is the tightening torque for an M12 stren...,tables,Kohler Generator Man,0.0,The tightening torque for an M12 strength clas...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-C
answers,510
calls/q,3.0
prompt_tokens/q,2435
completion_tokens/q,174
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),4.7
"cost per 1,000 questions ($)",0.47


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2760,5804502,752395,0.9566
claude-sonnet-5-5,1050,1848082,41789,4.1141
